# Minimal REINFORCE Agent for Flight School

This notebook adapts the original REINFORCE example to use **Bluebird Flight School** instead of `SectorIEnv`.

Flight School differs from the earlier example in an important way:

- it uses an **infinite traffic generator**
- the notebook uses the **centralized** control view
- the observation is one combined state vector
- the action is one discrete action integer each step

The notebook still includes:

- a small policy network
- REINFORCE training
- periodic evaluation during training
- comparison against a random baseline
- checkpoint saving and best-model restore
- final GIF rendering of the best policy

## Environment note

This notebook assumes you are already running the **correct Python/Jupyter kernel**:
one with `gymnasium`, `torch`, and the Bluebird project dependencies installed.

Do the package setup in your shell or project virtual environment first,
then open this notebook with that kernel. The notebook does **not** try to install
packages itself.

## Imports and path setup

This cell makes the notebook runnable from either the `bluebird-gymnasium` directory
or the repo root by adding the local package paths to `sys.path`.

In [ ]:
from __future__ import annotations

import random
import shutil
import sys
from pathlib import Path

import imageio
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import Image, display

search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
gym_root = None
dt_root = None

for candidate in search_roots:
    if (candidate / 'bluebird_gymnasium').exists():
        gym_root = candidate
        sibling_dt = candidate.parent / 'bluebird-dt'
        if sibling_dt.exists():
            dt_root = sibling_dt
        break
    if (candidate / 'bluebird-gymnasium').exists() and (candidate / 'bluebird-dt').exists():
        gym_root = candidate / 'bluebird-gymnasium'
        dt_root = candidate / 'bluebird-dt'
        break

if gym_root is None or dt_root is None:
    raise RuntimeError('Could not locate local bluebird-gymnasium and bluebird-dt package roots.')

sys.path.insert(0, str(gym_root))
sys.path.insert(0, str(dt_root))

from bluebird_gymnasium.envs import EnvConfig, ViewType
from bluebird_gymnasium.envs.flight_school import FlightSchoolEnv

print(f'Using bluebird-gymnasium from: {gym_root}')
print(f'Using bluebird-dt from: {dt_root}')
print(f'Torch version: {torch.__version__}')

## Policy network and agents

Because Flight School is used here in **centralized** mode:

- the input is one observation vector `obs`
- the policy outputs one set of action logits
- the chosen action is one integer

This makes the REINFORCE logic a little simpler than the decentralized notebook.

In [ ]:
class PolicyNetwork(nn.Module):
    """Neural network that maps one observation vector to action logits."""

    def __init__(
        self,
        observation_dimension: int,
        number_of_actions: int,
        hidden_units: int = 128,
    ) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(observation_dimension, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, number_of_actions),
        )

    def forward(self, observation_batch: torch.Tensor) -> torch.Tensor:
        return self.layers(observation_batch)


class SharedPolicyAgent:
    """Single policy network for centralized Flight School control."""

    def __init__(
        self,
        observation_dimension: int,
        number_of_actions: int,
        learning_rate: float = 1e-3,
        hidden_units: int = 128,
    ) -> None:
        self.policy_network = PolicyNetwork(
            observation_dimension=observation_dimension,
            number_of_actions=number_of_actions,
            hidden_units=hidden_units,
        )
        self.optimizer = optim.Adam(
            self.policy_network.parameters(),
            lr=learning_rate,
        )

    def sample_training_action(
        self,
        observation_vector: np.ndarray,
    ) -> tuple[int, torch.Tensor]:
        observation_tensor = torch.tensor(
            observation_vector,
            dtype=torch.float32,
        ).unsqueeze(0)
        action_logits = self.policy_network(observation_tensor)
        action_distribution = torch.distributions.Categorical(logits=action_logits)
        sampled_action = action_distribution.sample()
        action_log_probability = action_distribution.log_prob(sampled_action)
        return sampled_action.item(), action_log_probability.squeeze(0)

    def choose_evaluation_action(
        self,
        observation_vector: np.ndarray,
    ) -> int:
        with torch.no_grad():
            observation_tensor = torch.tensor(
                observation_vector,
                dtype=torch.float32,
            ).unsqueeze(0)
            action_logits = self.policy_network(observation_tensor)
            return torch.argmax(action_logits, dim=-1).item()

    def update_policy_from_episode(
        self,
        log_probability_per_step: list[torch.Tensor],
        reward_per_step: list[float],
        discount_factor_gamma: float = 0.99,
    ) -> float | None:
        if not log_probability_per_step:
            return None

        discounted_return_per_step: list[float] = []
        running_discounted_return = 0.0

        for reward in reversed(reward_per_step):
            running_discounted_return = reward + discount_factor_gamma * running_discounted_return
            discounted_return_per_step.append(running_discounted_return)

        discounted_return_per_step.reverse()
        returns_tensor = torch.tensor(discounted_return_per_step, dtype=torch.float32)

        if returns_tensor.numel() > 1:
            returns_std = returns_tensor.std(unbiased=False)
            if returns_std > 1e-8:
                returns_tensor = (
                    (returns_tensor - returns_tensor.mean())
                    / (returns_std + 1e-8)
                )

        policy_loss_terms: list[torch.Tensor] = []
        for action_log_probability, discounted_return in zip(
            log_probability_per_step,
            returns_tensor,
        ):
            policy_loss_terms.append(-action_log_probability * discounted_return)

        loss = torch.stack(policy_loss_terms).sum()
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()

    def save_checkpoint(self, checkpoint_path: Path, metadata: dict | None = None) -> None:
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            'policy_state_dict': self.policy_network.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'metadata': metadata or {},
        }
        torch.save(payload, checkpoint_path)

    def load_checkpoint(self, checkpoint_path: Path, map_location: str = 'cpu') -> dict:
        payload = torch.load(checkpoint_path, map_location=map_location)
        self.policy_network.load_state_dict(payload['policy_state_dict'])
        if 'optimizer_state_dict' in payload:
            self.optimizer.load_state_dict(payload['optimizer_state_dict'])
        return payload.get('metadata', {})


class RandomAgent:
    """Simple random baseline for centralized control."""

    def __init__(self, number_of_actions: int) -> None:
        self.number_of_actions = number_of_actions

    def choose_evaluation_action(self, observation_vector: np.ndarray) -> int:
        _ = observation_vector
        return random.randrange(self.number_of_actions)

## Flight School configuration and rollout helpers

This notebook uses `FlightSchoolEnv` in centralized mode.

The default Flight School config already includes:

- infinite traffic generation
- a shaped reward with safety terms
- a 10-minute Gymnasium episode horizon

The helper functions below support:

- one training episode
- one evaluation episode
- evaluation over many seeds
- GIF rendering for a final evaluation rollout

In [ ]:
def make_flight_school_training_config(random_seed: int | None = None) -> EnvConfig:
    config = FlightSchoolEnv.get_default_env_config(ViewType.CENTRALIZED)
    config.scenario_config['args']['random_seed'] = random_seed
    config.scenario_duration = 10 * 60
    config.view_config['type'] = ViewType.CENTRALIZED.value
    return config


def run_one_training_episode(
    environment: FlightSchoolEnv,
    agent: SharedPolicyAgent,
    random_seed: int,
    discount_factor_gamma: float,
) -> tuple[float, int, float | None]:
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)
    environment.config.scenario_config['args']['random_seed'] = random_seed

    observation_vector, _info = environment.reset(seed=random_seed)

    episode_is_done = False
    episode_step_count = 0
    episode_total_reward = 0.0

    log_probability_per_step: list[torch.Tensor] = []
    reward_per_step: list[float] = []

    while not episode_is_done:
        action_int, action_log_probability = agent.sample_training_action(observation_vector)
        (
            next_observation_vector,
            reward,
            done,
            truncated,
            _info,
        ) = environment.step(action_int)

        log_probability_per_step.append(action_log_probability)
        reward_per_step.append(float(reward))
        episode_total_reward += float(reward)
        episode_is_done = bool(done or truncated)
        observation_vector = next_observation_vector
        episode_step_count += 1

    loss_value = agent.update_policy_from_episode(
        log_probability_per_step=log_probability_per_step,
        reward_per_step=reward_per_step,
        discount_factor_gamma=discount_factor_gamma,
    )

    return episode_total_reward, episode_step_count, loss_value


def run_one_evaluation_episode(
    environment: FlightSchoolEnv,
    evaluation_agent,
    random_seed: int,
) -> tuple[float, int]:
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)
    environment.config.scenario_config['args']['random_seed'] = random_seed

    observation_vector, _info = environment.reset(seed=random_seed)

    episode_is_done = False
    episode_step_count = 0
    episode_total_reward = 0.0

    while not episode_is_done:
        action_int = evaluation_agent.choose_evaluation_action(observation_vector)
        (
            next_observation_vector,
            reward,
            done,
            truncated,
            _info,
        ) = environment.step(action_int)

        episode_total_reward += float(reward)
        episode_is_done = bool(done or truncated)
        observation_vector = next_observation_vector
        episode_step_count += 1

    return episode_total_reward, episode_step_count


def evaluate_agent_over_seeds(
    environment: FlightSchoolEnv,
    evaluation_agent,
    evaluation_seeds: list[int],
) -> dict:
    rewards: list[float] = []
    steps: list[int] = []

    for random_seed in evaluation_seeds:
        total_reward, step_count = run_one_evaluation_episode(
            environment=environment,
            evaluation_agent=evaluation_agent,
            random_seed=random_seed,
        )
        rewards.append(total_reward)
        steps.append(step_count)

    return {
        'seeds': evaluation_seeds,
        'rewards': rewards,
        'steps': steps,
        'mean_reward': float(np.mean(rewards)),
        'std_reward': float(np.std(rewards)),
        'mean_steps': float(np.mean(steps)),
    }


def render_evaluation_rollout_to_gif(
    agent: SharedPolicyAgent,
    random_seed: int,
    render_dir: Path,
    gif_name: str = 'flight_school_reinforce_eval',
    render_every_n_steps: int = 5,
    gif_frame_duration_seconds: float = 0.2,
) -> Path:
    render_config = make_flight_school_training_config(random_seed=random_seed)
    render_config.radar_config['display_actions'] = True
    render_config.radar_config['render_dir'] = str(render_dir)
    render_config.radar_config['prefix'] = 'frame'

    if render_dir.exists():
        shutil.rmtree(render_dir)
    render_dir.mkdir(parents=True, exist_ok=True)

    render_environment = FlightSchoolEnv(config=render_config)
    render_environment.set_render_mode('file')

    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)

    observation_vector, _info = render_environment.reset(seed=random_seed)
    render_environment.render()

    episode_is_done = False
    step_index = 0

    while not episode_is_done:
        action_int = agent.choose_evaluation_action(observation_vector)
        (
            next_observation_vector,
            reward,
            done,
            truncated,
            _info,
        ) = render_environment.step(action_int)
        _ = reward
        step_index += 1
        episode_is_done = bool(done or truncated)
        if step_index % render_every_n_steps == 0 or episode_is_done:
            render_environment.render()
        observation_vector = next_observation_vector

    png_frames = sorted(render_dir.glob(f"{render_config.radar_config['prefix']}_*.png"))
    if not png_frames:
        raise RuntimeError(
            f'No rendered PNG frames were written to {render_dir}. '
            'Expected at least one frame before GIF generation.'
        )

    gif_path = render_dir / f'{gif_name}.gif'
    images = [imageio.v3.imread(frame_path) for frame_path in png_frames]
    imageio.mimsave(gif_path, images, loop=0, duration=gif_frame_duration_seconds)
    render_environment.close()
    return gif_path


## Set up the environment and inspect the shapes

For this notebook, the most important values are:

- `observation_dimension`: how many numbers are in the centralized observation vector
- `number_of_actions`: how many discrete actions the policy can choose from

In [ ]:
initial_seed = 7
config = make_flight_school_training_config(random_seed=initial_seed)
environment = FlightSchoolEnv(config=config)

observation_dimension = environment.observation_space.shape[0]
number_of_actions = environment.action_space.n

print(
    'environment shapes:',
    f'observation_dimension={observation_dimension}',
    f'number_of_actions={number_of_actions}',
)

## Hyperparameters and experiment settings

This version keeps the same richer experiment loop:

- periodic evaluation during training
- larger held-out evaluation set
- checkpoint saving
- automatic tracking of the best evaluated model
- random-policy baseline comparison

In [ ]:
learning_rate = 1e-3
hidden_units = 128
discount_factor_gamma = 0.99
number_of_training_episodes = 100
training_seed_start = 100
periodic_eval_interval = 10
heldout_evaluation_seeds = list(range(200, 220))
checkpoint_dir = Path.cwd() / 'checkpoints' / 'minimal_reinforce_flight_school'
latest_checkpoint_path = checkpoint_dir / 'latest.pt'
best_checkpoint_path = checkpoint_dir / 'best.pt'

agent = SharedPolicyAgent(
    observation_dimension=observation_dimension,
    number_of_actions=number_of_actions,
    learning_rate=learning_rate,
    hidden_units=hidden_units,
)
random_agent = RandomAgent(number_of_actions=number_of_actions)

training_rewards: list[float] = []
training_steps: list[int] = []
training_losses: list[float] = []

periodic_eval_episodes: list[int] = []
periodic_eval_learned_mean_rewards: list[float] = []
periodic_eval_learned_std_rewards: list[float] = []
periodic_eval_random_mean_rewards: list[float] = []
periodic_eval_random_std_rewards: list[float] = []
periodic_eval_learned_mean_steps: list[float] = []
periodic_eval_random_mean_steps: list[float] = []

best_mean_evaluation_reward = float('-inf')
best_checkpoint_metadata: dict = {}
render_every_n_steps = 5
gif_frame_duration_seconds = 0.2


## Training loop with periodic evaluation and checkpointing

Every `periodic_eval_interval` episodes, the notebook:

- evaluates the current learned policy on the held-out evaluation seeds
- evaluates a random baseline on the same seeds
- saves a `latest.pt` checkpoint
- overwrites `best.pt` if the learned policy achieves a new best mean evaluation reward

In [ ]:
for episode_index in range(number_of_training_episodes):
    random_seed = training_seed_start + episode_index
    total_reward, step_count, loss_value = run_one_training_episode(
        environment=environment,
        agent=agent,
        random_seed=random_seed,
        discount_factor_gamma=discount_factor_gamma,
    )

    training_rewards.append(total_reward)
    training_steps.append(step_count)
    training_losses.append(float('nan') if loss_value is None else loss_value)

    print(
        '[train]',
        f'episode={episode_index:03d}',
        f'seed={random_seed}',
        f'reward={total_reward:.3f}',
        f'steps={step_count}',
        f'loss={loss_value}',
    )

    should_run_periodic_eval = (
        (episode_index + 1) % periodic_eval_interval == 0
        or episode_index == number_of_training_episodes - 1
    )

    if should_run_periodic_eval:
        learned_eval = evaluate_agent_over_seeds(
            environment=environment,
            evaluation_agent=agent,
            evaluation_seeds=heldout_evaluation_seeds,
        )
        random_eval = evaluate_agent_over_seeds(
            environment=environment,
            evaluation_agent=random_agent,
            evaluation_seeds=heldout_evaluation_seeds,
        )

        periodic_eval_episodes.append(episode_index + 1)
        periodic_eval_learned_mean_rewards.append(learned_eval['mean_reward'])
        periodic_eval_learned_std_rewards.append(learned_eval['std_reward'])
        periodic_eval_random_mean_rewards.append(random_eval['mean_reward'])
        periodic_eval_random_std_rewards.append(random_eval['std_reward'])
        periodic_eval_learned_mean_steps.append(learned_eval['mean_steps'])
        periodic_eval_random_mean_steps.append(random_eval['mean_steps'])

        metadata = {
            'episode': episode_index + 1,
            'train_seed': random_seed,
            'learned_mean_reward': learned_eval['mean_reward'],
            'learned_std_reward': learned_eval['std_reward'],
            'random_mean_reward': random_eval['mean_reward'],
            'random_std_reward': random_eval['std_reward'],
            'evaluation_seeds': heldout_evaluation_seeds,
        }
        agent.save_checkpoint(latest_checkpoint_path, metadata=metadata)

        if learned_eval['mean_reward'] > best_mean_evaluation_reward:
            best_mean_evaluation_reward = learned_eval['mean_reward']
            best_checkpoint_metadata = metadata
            agent.save_checkpoint(best_checkpoint_path, metadata=metadata)
            checkpoint_note = 'new best checkpoint'
        else:
            checkpoint_note = 'latest checkpoint only'

        print(
            '[periodic-eval]',
            f'episode={episode_index + 1:03d}',
            f'learned_mean_reward={learned_eval["mean_reward"]:.3f}',
            f'learned_std_reward={learned_eval["std_reward"]:.3f}',
            f'random_mean_reward={random_eval["mean_reward"]:.3f}',
            f'random_std_reward={random_eval["std_reward"]:.3f}',
            checkpoint_note,
        )

## Restore the best evaluated model

The training loop may end on a policy that is not the best one seen so far.
This cell reloads the checkpoint with the highest held-out mean evaluation reward.

In [ ]:
if not best_checkpoint_path.exists():
    raise FileNotFoundError(f'Best checkpoint not found: {best_checkpoint_path}')

loaded_metadata = agent.load_checkpoint(best_checkpoint_path)
print('Reloaded best checkpoint from:', best_checkpoint_path)
print('Best checkpoint metadata:')
loaded_metadata

## Final evaluation of the best checkpoint vs random baseline

This uses the larger held-out evaluation set and compares:

- the best learned policy checkpoint
- a random baseline on the same seeds

In [ ]:
best_policy_eval = evaluate_agent_over_seeds(
    environment=environment,
    evaluation_agent=agent,
    evaluation_seeds=heldout_evaluation_seeds,
)
random_policy_eval = evaluate_agent_over_seeds(
    environment=environment,
    evaluation_agent=random_agent,
    evaluation_seeds=heldout_evaluation_seeds,
)

print('Best policy evaluation mean reward:', best_policy_eval['mean_reward'])
print('Best policy evaluation std reward:', best_policy_eval['std_reward'])
print('Random baseline mean reward:', random_policy_eval['mean_reward'])
print('Random baseline std reward:', random_policy_eval['std_reward'])

## Useful plots

These are the most useful quick-look plots for this Flight School REINFORCE setup:

- **Training reward**: raw reward and moving average during training
- **Episode length**: how long training episodes run
- **Periodic evaluation**: learned policy vs random baseline over training
- **Final evaluation by seed**: best learned checkpoint vs random on the held-out seeds

In [ ]:
def moving_average(values: list[float], window: int) -> np.ndarray:
    if len(values) < window:
        return np.array([])
    kernel = np.ones(window) / window
    return np.convolve(np.asarray(values, dtype=float), kernel, mode='valid')

plot_window = min(10, len(training_rewards))
smoothed_rewards = moving_average(training_rewards, plot_window)
training_episode_indices = np.arange(1, len(training_rewards) + 1)
heldout_seed_indices = np.arange(len(heldout_evaluation_seeds))
periodic_eval_episodes_arr = np.asarray(periodic_eval_episodes)
learned_mean_arr = np.asarray(periodic_eval_learned_mean_rewards)
learned_std_arr = np.asarray(periodic_eval_learned_std_rewards)
random_mean_arr = np.asarray(periodic_eval_random_mean_rewards)
random_std_arr = np.asarray(periodic_eval_random_std_rewards)

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

axes[0, 0].plot(training_episode_indices, training_rewards, marker='o', alpha=0.25, label='raw reward')
if len(smoothed_rewards) > 0:
    axes[0, 0].plot(
        np.arange(plot_window, len(training_rewards) + 1),
        smoothed_rewards,
        linewidth=2.5,
        color='tab:blue',
        label=f'moving average (window={plot_window})',
    )
axes[0, 0].set_title('Training Reward per Episode')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Total Reward')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(training_episode_indices, training_steps, marker='o', color='tab:orange')
axes[0, 1].set_title('Training Episode Length')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Steps')
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(periodic_eval_episodes_arr, learned_mean_arr, marker='o', label='learned policy')
axes[1, 0].fill_between(
    periodic_eval_episodes_arr,
    learned_mean_arr - learned_std_arr,
    learned_mean_arr + learned_std_arr,
    alpha=0.2,
)
axes[1, 0].plot(periodic_eval_episodes_arr, random_mean_arr, marker='s', label='random baseline')
axes[1, 0].fill_between(
    periodic_eval_episodes_arr,
    random_mean_arr - random_std_arr,
    random_mean_arr + random_std_arr,
    alpha=0.2,
)
axes[1, 0].set_title('Periodic Evaluation: Learned vs Random')
axes[1, 0].set_xlabel('Training Episode')
axes[1, 0].set_ylabel('Mean Evaluation Reward')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(
    heldout_seed_indices,
    best_policy_eval['rewards'],
    marker='o',
    linewidth=2,
    label='best learned checkpoint',
)
axes[1, 1].plot(
    heldout_seed_indices,
    random_policy_eval['rewards'],
    marker='s',
    linewidth=2,
    label='random baseline',
)
axes[1, 1].set_xticks(heldout_seed_indices)
axes[1, 1].set_xticklabels([str(seed) for seed in heldout_evaluation_seeds], rotation=45)
axes[1, 1].set_title('Final Evaluation Reward by Seed')
axes[1, 1].set_xlabel('Held-out Evaluation Seed')
axes[1, 1].set_ylabel('Total Reward')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

fig.suptitle('Flight School REINFORCE Summary with Baseline and Checkpoints', fontsize=16)
fig.tight_layout()
plt.show()

print(f'Best checkpoint mean evaluation reward: {best_policy_eval["mean_reward"]:.3f}')
print(f'Random baseline mean evaluation reward: {random_policy_eval["mean_reward"]:.3f}')
print(f'Latest checkpoint path: {latest_checkpoint_path}')
print(f'Best checkpoint path: {best_checkpoint_path}')

## Render the best checkpoint and save a GIF

This section runs the **best restored checkpoint** in evaluation mode with Bluebird radar rendering enabled.
It saves individual frames to disk and then combines them into a GIF.

Notes:

- `display_actions=True` overlays actions on the radar frames
- the GIF path is printed and displayed in the notebook
- by default this uses the first held-out evaluation seed
- `render_every_n_steps` controls how many simulator steps to skip between saved frames
- `gif_frame_duration_seconds` controls how long each GIF frame is shown


In [ ]:
gif_seed = heldout_evaluation_seeds[0]
render_dir = Path.cwd() / 'renders' / 'minimal_reinforce_flight_school_eval'
gif_path = render_evaluation_rollout_to_gif(
    agent=agent,
    random_seed=gif_seed,
    render_dir=render_dir,
    gif_name=f'best_checkpoint_eval_seed_{gif_seed}',
    render_every_n_steps=render_every_n_steps,
    gif_frame_duration_seconds=gif_frame_duration_seconds,
)

print(f'Saved GIF to: {gif_path}')
display(Image(filename=str(gif_path)))

## Optional cleanup

Close the main environment if you are done with the notebook session.

In [ ]:
environment.close()